# STS Ewe↔Fr — Entraînement + Évaluation + Analyse (Notebook)

Objectif: exécuter une expérience (train/val/test), calculer **BLEU/chrF/TER** (MT) et analyser les erreurs à partir d’exports **CSV/JSON**.

Ce notebook est conçu pour fonctionner en 2 modes:
1) **Mode “j’ai déjà des checkpoints / résultats”**: charge `docs/results/*.json` et `docs/results/*.csv` et produit l’analyse.
2) **Mode “j’entraîne ici”**: entraîne un modèle Seq2Seq (Transformers) sur `data/manifests/mt/manifest_ewe_fr_bitext.tsv`, génère des prédictions val/test, exporte tout dans `docs/results/runs/<run_id>/`.

Prérequis: Python 3.12 (venv `F:/STS/.venv`). GPU fortement recommandé pour l’entraînement.

## 1) Installer / importer les dépendances

Si besoin, décommente l’installation. Sur Windows, privilégie l’installation dans `.venv`.

Packages: `transformers`, `datasets`, `torch`, `accelerate`, `evaluate`, `sacrebleu`, `sentencepiece`, `pandas`, `numpy`, `matplotlib`, `seaborn`.

In [ ]:
# Optional: install deps (run once)
# %pip install -U pip
# %pip install torch transformers accelerate datasets evaluate sacrebleu sentencepiece pandas numpy matplotlib seaborn

import os
import json
import time
import math
import hashlib
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Repro
import random

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

seed_everything(42)

# CUDA / versions
try:
    import torch
    print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch not available:", e)

try:
    import transformers
    import datasets
    import evaluate
    print("transformers:", transformers.__version__)
    print("datasets:", datasets.__version__)
    print("evaluate:", evaluate.__version__)
except Exception as e:
    print("HF stack not available:", e)

ROOT = Path("..").resolve()  # notebooks/ -> repo root
print("repo root:", ROOT)


## 2) Définir la config d’expérience (chemins, seed, hyperparamètres, artefacts)

On va écrire tous les artefacts dans `docs/results/runs/<run_id>/`.

- `config_run.json`
- `train_log.csv`
- `metrics_val.json` / `metrics_test.json` (+ `.csv`)
- `predictions_val.csv` / `predictions_test.csv` (+ `.jsonl`)
- `error_analysis.csv`

In [ ]:
@dataclass
class RunConfig:
    run_name: str = "mt_nllb_ee_fr"
    direction: str = "ee_to_fr"  # ee_to_fr | fr_to_ee
    seed: int = 42

    manifest_path: str = "data/manifests/mt/manifest_ewe_fr_bitext.tsv"

    base_model: str = "facebook/nllb-200-distilled-600M"
    max_source_length: int = 256
    max_target_length: int = 256

    # training
    per_device_train_batch_size: int = 8
    per_device_eval_batch_size: int = 8
    gradient_accumulation_steps: int = 2
    learning_rate: float = 2e-5
    num_train_epochs: float = 1.0
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01

    # generation
    num_beams: int = 4
    length_penalty: float = 1.0
    max_new_tokens: int = 128
    no_repeat_ngram_size: int = 0

    # data sizing (quick mode)
    max_train: int = 20000
    max_val: int = 2000
    max_test: int = 2000

    # exports
    out_root: str = "docs/results/runs"

cfg = RunConfig()
seed_everything(cfg.seed)

# run_id = hash of config + timestamp
cfg_json = json.dumps(asdict(cfg), ensure_ascii=False, sort_keys=True)
run_hash = hashlib.sha1(cfg_json.encode("utf-8")).hexdigest()[:10]
run_id = f"{cfg.run_name}_{run_hash}_{time.strftime('%Y%m%d_%H%M%S')}"

RUN_DIR = ROOT / cfg.out_root / run_id
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Save config
(RUN_DIR / "config_run.json").write_text(cfg_json, encoding="utf-8")
print("run dir:", RUN_DIR)


## 3) Charger les données + split train/val/test (reproductible)

Source: `data/manifests/mt/manifest_ewe_fr_bitext.tsv` avec colonnes attendues: `split`, `ee`, `fr`.

On respecte les splits déjà présents (`train/dev/test`).

In [ ]:
manifest_path = ROOT / cfg.manifest_path
assert manifest_path.exists(), f"Missing: {manifest_path}"

df = pd.read_csv(manifest_path, sep="\t")
expected = {"split", "ee", "fr"}
missing = expected - set(df.columns)
assert not missing, f"Missing columns: {missing}"

print("rows:", len(df))
print(df["split"].value_counts())

# pick direction
if cfg.direction == "ee_to_fr":
    SRC_COL, TGT_COL = "ee", "fr"
else:
    SRC_COL, TGT_COL = "fr", "ee"

# keep only non-empty
df = df[["split", SRC_COL, TGT_COL]].rename(columns={SRC_COL: "src", TGT_COL: "tgt"})
df["src"] = df["src"].astype(str)
df["tgt"] = df["tgt"].astype(str)
df = df[(df["src"].str.strip() != "") & (df["tgt"].str.strip() != "")]

train_df = df[df.split == "train"].head(cfg.max_train)
val_df   = df[df.split == "dev"].head(cfg.max_val)
test_df  = df[df.split == "test"].head(cfg.max_test)

sizes = {"train": len(train_df), "val": len(val_df), "test": len(test_df)}
(RUN_DIR / "data_split_sizes.json").write_text(json.dumps(sizes, indent=2), encoding="utf-8")
print(sizes)

# quick peek
train_df.head(3)

## 4) Nettoyage / normalisation texte + contrôles qualité

Nettoyage minimal (baseline):
- trim + normalisation espaces
- suppression lignes vides
- filtrage par longueurs extrêmes
- déduplication exact match (src,tgt)

On exporte un rapport qualité: distributions de longueurs (chars + approx tokens).

In [ ]:
import re

_ws = re.compile(r"\s+")

def normalize_text(s: str) -> str:
    s = str(s)
    s = s.replace("\u00A0", " ")
    s = _ws.sub(" ", s).strip()
    return s

def clean_df(df_in: pd.DataFrame, min_chars: int = 1, max_chars: int = 1000) -> pd.DataFrame:
    dfc = df_in.copy()
    dfc["src"] = dfc["src"].map(normalize_text)
    dfc["tgt"] = dfc["tgt"].map(normalize_text)
    dfc = dfc[(dfc.src.str.len() >= min_chars) & (dfc.tgt.str.len() >= min_chars)]
    dfc = dfc[(dfc.src.str.len() <= max_chars) & (dfc.tgt.str.len() <= max_chars)]
    dfc = dfc.drop_duplicates(subset=["src", "tgt"], keep="first")
    return dfc.reset_index(drop=True)

train_df_c = clean_df(train_df)
val_df_c = clean_df(val_df)
test_df_c = clean_df(test_df)

print({"train": (len(train_df), len(train_df_c)), "val": (len(val_df), len(val_df_c)), "test": (len(test_df), len(test_df_c))})

def quality_report(df_in: pd.DataFrame, name: str) -> dict:
    src_len = df_in.src.str.len().to_numpy()
    tgt_len = df_in.tgt.str.len().to_numpy()
    rep = {
        "name": name,
        "n": int(len(df_in)),
        "src_chars": {
            "min": int(src_len.min()) if len(src_len) else 0,
            "p50": float(np.percentile(src_len, 50)) if len(src_len) else 0.0,
            "p95": float(np.percentile(src_len, 95)) if len(src_len) else 0.0,
            "max": int(src_len.max()) if len(src_len) else 0,
        },
        "tgt_chars": {
            "min": int(tgt_len.min()) if len(tgt_len) else 0,
            "p50": float(np.percentile(tgt_len, 50)) if len(tgt_len) else 0.0,
            "p95": float(np.percentile(tgt_len, 95)) if len(tgt_len) else 0.0,
            "max": int(tgt_len.max()) if len(tgt_len) else 0,
        },
    }
    return rep

qr = {
    "train": quality_report(train_df_c, "train"),
    "val": quality_report(val_df_c, "val"),
    "test": quality_report(test_df_c, "test"),
}
(RUN_DIR / "quality_report.json").write_text(json.dumps(qr, ensure_ascii=False, indent=2), encoding="utf-8")
qr

## 5) Tokenisation + préparation des batches (DataCollator)

On utilise un tokenizer NLLB, et on prépare des datasets `transformers` pour `Seq2SeqTrainer`.

In [ ]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    DataCollatorForSeq2Seq,
)

tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, use_fast=True)

# NLLB language tags for Ewe/French
NLLB_EWE = "ewe_Latn"
NLLB_FR  = "fra_Latn"

if cfg.direction == "ee_to_fr":
    src_lang = NLLB_EWE
    tgt_lang = NLLB_FR
else:
    src_lang = NLLB_FR
    tgt_lang = NLLB_EWE

# Some tokenizers expose these attributes
if hasattr(tokenizer, "src_lang"):
    tokenizer.src_lang = src_lang

train_ds = Dataset.from_pandas(train_df_c[["src", "tgt"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df_c[["src", "tgt"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df_c[["src", "tgt"]], preserve_index=False)

def preprocess(batch):
    model_inputs = tokenizer(
        batch["src"],
        max_length=cfg.max_source_length,
        truncation=True,
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["tgt"],
            max_length=cfg.max_target_length,
            truncation=True,
        )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=["src", "tgt"])
val_tok   = val_ds.map(preprocess, batched=True, remove_columns=["src", "tgt"])
test_tok  = test_ds.map(preprocess, batched=True, remove_columns=["src", "tgt"])

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=None)

# sanity check
train_tok[0]


## 6) Initialiser le modèle Seq2Seq + tokenizer

On charge `facebook/nllb-200-distilled-600M`. Pour NLLB, on force souvent la langue cible via `forced_bos_token_id`.

Astuce: pour accélérer, tu peux activer `gradient_checkpointing` (au prix de compute) si nécessaire.

In [ ]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(cfg.base_model)

# Force target language for NLLB
try:
    forced_bos = tokenizer.convert_tokens_to_ids(tgt_lang)
    model.config.forced_bos_token_id = forced_bos
    print("forced_bos_token_id:", forced_bos, "tgt_lang:", tgt_lang)
except Exception as e:
    print("Could not set forced_bos_token_id:", e)

# Optional
# model.gradient_checkpointing_enable()

# Save resolved model/tokenizer info
resolved = {
    "base_model": cfg.base_model,
    "direction": cfg.direction,
    "src_lang": src_lang,
    "tgt_lang": tgt_lang,
    "forced_bos_token_id": getattr(model.config, "forced_bos_token_id", None),
}
(RUN_DIR / "model_info.json").write_text(json.dumps(resolved, ensure_ascii=False, indent=2), encoding="utf-8")
resolved

## 7) Entraînement complet (checkpoints, early stopping, logs)

On utilise `Seq2SeqTrainer` pour générer pendant l’évaluation et enregistrer logs/checkpoints.

Note: pour un run *rapide* de validation, réduis `cfg.max_train/cfg.max_val` et `cfg.num_train_epochs`.

In [ ]:
from transformers import (
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)

# Metrics (corpus-level) during eval
import evaluate
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def postprocess_text(preds, labels):
    preds = [p.strip() for p in preds]
    labels = [l.strip() for l in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    bleu = bleu_metric.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])["score"]
    chrf = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels)["score"]

    pred_lens = [len(p.split()) for p in decoded_preds]
    label_lens = [len(l.split()) for l in decoded_labels]
    length_ratio = float(np.mean([(pl / ll) if ll else 0.0 for pl, ll in zip(pred_lens, label_lens)]))

    return {
        "bleu": float(bleu),
        "chrf": float(chrf),
        "len_ratio": length_ratio,
    }

TRAIN_DIR = RUN_DIR / "checkpoints"
LOG_CSV = RUN_DIR / "train_log.csv"

args = Seq2SeqTrainingArguments(
    output_dir=str(TRAIN_DIR),
    overwrite_output_dir=True,
    seed=cfg.seed,

    per_device_train_batch_size=cfg.per_device_train_batch_size,
    per_device_eval_batch_size=cfg.per_device_eval_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,

    learning_rate=cfg.learning_rate,
    num_train_epochs=cfg.num_train_epochs,
    warmup_ratio=cfg.warmup_ratio,
    weight_decay=cfg.weight_decay,

    logging_strategy="steps",
    logging_steps=50,

    evaluation_strategy="steps",
    eval_steps=500,

    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    predict_with_generate=True,
    generation_num_beams=cfg.num_beams,
    generation_max_new_tokens=cfg.max_new_tokens,

    load_best_model_at_end=True,
    metric_for_best_model="chrf",
    greater_is_better=True,

    report_to=[],  # no wandb by default
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# Optional: resume
# trainer.train(resume_from_checkpoint=True)

train_result = trainer.train()
trainer.save_model(str(RUN_DIR / "model"))

# Save train logs
logs = trainer.state.log_history
pd.DataFrame(logs).to_csv(LOG_CSV, index=False)
print("saved:", LOG_CSV)
train_result

## 8) Génération sur val/test (beam search, paramètres de décodage)

On exporte pour chaque exemple: `id`, `source`, `reference`, `hypothesis`, `len_ratio`.

Ces exports servent ensuite à l’analyse d’erreurs et aux visualisations.

In [ ]:
def predict_split(raw_df: pd.DataFrame, split_name: str):
    # Ensure we use the trained model saved in RUN_DIR/model
    from transformers import AutoModelForSeq2SeqLM

    model_dir = RUN_DIR / "model"
    assert model_dir.exists(), "Train first or point to an existing model dir"

    model_inf = AutoModelForSeq2SeqLM.from_pretrained(str(model_dir))

    # keep same forced_bos if available
    try:
        forced_bos = tokenizer.convert_tokens_to_ids(tgt_lang)
        model_inf.config.forced_bos_token_id = forced_bos
    except Exception:
        pass

    device = "cuda" if ("torch" in globals() and torch.cuda.is_available()) else "cpu"
    model_inf.to(device)
    model_inf.eval()

    sources = raw_df["src"].tolist()
    refs = raw_df["tgt"].tolist()

    hyps = []
    for i in range(0, len(sources), cfg.per_device_eval_batch_size):
        batch = sources[i : i + cfg.per_device_eval_batch_size]
        enc = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=cfg.max_source_length,
        ).to(device)
        with torch.no_grad():
            out = model_inf.generate(
                **enc,
                num_beams=cfg.num_beams,
                length_penalty=cfg.length_penalty,
                max_new_tokens=cfg.max_new_tokens,
                no_repeat_ngram_size=cfg.no_repeat_ngram_size,
            )
        hyp = tokenizer.batch_decode(out, skip_special_tokens=True)
        hyps.extend([h.strip() for h in hyp])

    out_df = pd.DataFrame(
        {
            "id": list(range(len(sources))),
            "split": split_name,
            "source": sources,
            "reference": refs,
            "hypothesis": hyps,
        }
    )
    out_df["len_ratio_chars"] = (out_df.hypothesis.str.len() / out_df.reference.str.len().replace(0, np.nan)).fillna(0.0)
    return out_df

pred_val = predict_split(val_df_c, "val")
pred_test = predict_split(test_df_c, "test")

# Export stable schema
pred_val.to_csv(RUN_DIR / "predictions_val.csv", index=False)
pred_test.to_csv(RUN_DIR / "predictions_test.csv", index=False)

pred_val.to_json(RUN_DIR / "predictions_val.jsonl", orient="records", lines=True, force_ascii=False)
pred_test.to_json(RUN_DIR / "predictions_test.jsonl", orient="records", lines=True, force_ascii=False)

print("saved preds:", RUN_DIR)
pred_test.head(3)

## 9) Évaluation automatique (BLEU, chrF, TER + stats auxiliaires)

On calcule les métriques corpus avec `sacrebleu`.

Formules (rappel):

$$BP = \exp(1-\frac{r}{c})\;\text{si}\;c<r;\quad BP=1\;\text{sinon}$$
$$BLEU = BP \cdot \exp\left(\sum_{n=1}^{N} w_n\log p_n\right)$$

où $c$ est la longueur candidate (hypothèse) et $r$ la longueur référence, $p_n$ les précisions n-gram.

In [ ]:
import sacrebleu

def corpus_metrics(pred_df: pd.DataFrame) -> dict:
    hyps = pred_df["hypothesis"].astype(str).tolist()
    refs = pred_df["reference"].astype(str).tolist()

    bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
    chrf = sacrebleu.corpus_chrf(hyps, [refs]).score
    ter  = sacrebleu.corpus_ter(hyps, [refs]).score

    # Aux stats
    hyp_len = pred_df.hypothesis.str.len().to_numpy()
    ref_len = pred_df.reference.str.len().to_numpy()
    len_ratio = float(np.mean((hyp_len / np.maximum(ref_len, 1))))

    return {
        "n": int(len(pred_df)),
        "bleu": float(bleu),
        "chrf": float(chrf),
        "ter": float(ter),
        "len_ratio_chars": len_ratio,
    }

metrics_val = corpus_metrics(pred_val)
metrics_test = corpus_metrics(pred_test)

# Save metrics JSON/CSV
(RUN_DIR / "metrics_val.json").write_text(json.dumps(metrics_val, indent=2), encoding="utf-8")
(RUN_DIR / "metrics_test.json").write_text(json.dumps(metrics_test, indent=2), encoding="utf-8")

pd.DataFrame([{"split": "val", **metrics_val}, {"split": "test", **metrics_test}]).to_csv(RUN_DIR / "metrics.csv", index=False)
metrics_test

## 10) Sauvegarder résultats en CSV/JSON (métriques, config, prédictions, erreurs)

Ici, on a:
- `config_run.json`
- `train_log.csv`
- `metrics_val.json`, `metrics_test.json`, `metrics.csv`
- `predictions_val.csv/jsonl`, `predictions_test.csv/jsonl`

On ajoute aussi un fichier `summary.json` pratique pour comparer des runs.

In [ ]:
summary = {
    "run_id": run_id,
    **asdict(cfg),
    "metrics_val": metrics_val,
    "metrics_test": metrics_test,
}
(RUN_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved summary.json")
summary

## 11) Analyse des erreurs

Approche baseline (rapide, utile académiquement):
- score phrase-level via `sacrebleu.sentence_chrf`
- buckets par longueur source
- détecteurs heuristiques: sur/sous-traduction via `len_ratio_chars`, répétitions simples, copie partielle.

On exporte `error_analysis.csv`.

In [ ]:
def repetition_score(text: str, n: int = 3) -> float:
    toks = text.split()
    if len(toks) < n:
        return 0.0
    grams = [" ".join(toks[i : i + n]) for i in range(len(toks) - n + 1)]
    if not grams:
        return 0.0
    uniq = len(set(grams))
    return 1.0 - (uniq / len(grams))

def add_error_features(pred_df: pd.DataFrame) -> pd.DataFrame:
    out = pred_df.copy()
    out["chrf_sentence"] = [
        sacrebleu.sentence_chrf(h, [r]).score for h, r in zip(out.hypothesis.astype(str), out.reference.astype(str))
    ]
    out["len_src"] = out.source.astype(str).str.len()
    out["len_ref"] = out.reference.astype(str).str.len()
    out["len_hyp"] = out.hypothesis.astype(str).str.len()
    out["len_ratio"] = (out.len_hyp / out.len_ref.replace(0, np.nan)).fillna(0.0)

    out["rep3"] = out.hypothesis.astype(str).map(lambda s: repetition_score(s, 3))
    out["rep4"] = out.hypothesis.astype(str).map(lambda s: repetition_score(s, 4))

    out["over_translation"] = out.len_ratio > 1.4
    out["under_translation"] = out.len_ratio < 0.7
    out["high_repetition"] = out.rep3 > 0.4
    return out

err_test = add_error_features(pred_test)

# Buckets by source length
bins = [0, 40, 80, 120, 200, 400, 10_000]
labels = ["0-40", "40-80", "80-120", "120-200", "200-400", ">400"]
err_test["src_len_bucket"] = pd.cut(err_test.len_src, bins=bins, labels=labels, include_lowest=True)

bucket_metrics = (
    err_test.groupby("src_len_bucket", dropna=False)
    .agg(n=("id", "count"), chrf_mean=("chrf_sentence", "mean"), len_ratio_mean=("len_ratio", "mean"))
    .reset_index()
)

# Worst examples
worst = err_test.sort_values("chrf_sentence").head(50)

# Export
err_out = err_test.sort_values("chrf_sentence").reset_index(drop=True)
err_out.to_csv(RUN_DIR / "error_analysis.csv", index=False)

bucket_metrics.to_csv(RUN_DIR / "bucket_metrics.csv", index=False)

print("saved:", RUN_DIR / "error_analysis.csv")
bucket_metrics

## 12) Visualisations depuis CSV/JSON

Graphes utiles:
- courbes `loss/train` vs `eval_loss`
- distribution `len_ratio`
- `chrF` phrase vs longueur
- top erreurs (chrF le plus bas)
- comparaison de plusieurs runs (si tu as plusieurs dossiers dans `docs/results/runs/`)

In [ ]:
# 1) train logs (if present)
log_path = RUN_DIR / "train_log.csv"
if log_path.exists():
    logs_df = pd.read_csv(log_path)
    display(logs_df.tail())

    plt.figure(figsize=(10, 4))
    # Trainer logs may contain different fields across steps
    if "loss" in logs_df.columns:
        plt.plot(logs_df["step"], logs_df["loss"], label="train_loss", alpha=0.7)
    if "eval_loss" in logs_df.columns:
        plt.plot(logs_df["step"], logs_df["eval_loss"], label="eval_loss", alpha=0.9)
    plt.legend(); plt.title("Training / Eval Loss"); plt.xlabel("step"); plt.ylabel("loss")
    plt.show()
else:
    print("No train_log.csv in", RUN_DIR)

# 2) len ratio distribution
plt.figure(figsize=(8, 4))
sns.histplot(err_test["len_ratio"], bins=50)
plt.title("Length ratio (hyp/ref)")
plt.xlabel("len_ratio")
plt.show()

# 3) chrF sentence vs src length
plt.figure(figsize=(8, 4))
sns.scatterplot(data=err_test.sample(min(len(err_test), 2000), random_state=cfg.seed), x="len_src", y="chrf_sentence", alpha=0.35)
plt.title("chrF(sentence) vs source length")
plt.xlabel("len_src (chars)")
plt.ylabel("chrF(sentence)")
plt.show()

# 4) bucket metrics
plt.figure(figsize=(9, 4))
sns.barplot(data=bucket_metrics, x="src_len_bucket", y="chrf_mean")
plt.title("chrF mean by source length bucket")
plt.xlabel("bucket"); plt.ylabel("mean chrF")
plt.xticks(rotation=30)
plt.show()

# 5) worst examples table
cols = ["id", "chrf_sentence", "len_ratio", "high_repetition", "over_translation", "under_translation", "source", "reference", "hypothesis"]
display(worst[cols].head(10))


## 13) Synthèse actionnable: prioriser les points à améliorer

Règles simples (à adapter) basées sur constats:
- Sur-traduction fréquente → baisser `max_new_tokens`, augmenter `length_penalty`.
- Sous-traduction fréquente → augmenter `max_new_tokens`, réduire `length_penalty`.
- Répétitions → `no_repeat_ngram_size` (2–4) + vérifier le tokenizer/lang forcing.
- Perf faible sur phrases longues → augmenter `max_source_length`, entraînement plus long, curriculum, batch/accum.
- Vocab rare / erreurs systématiques → plus de données ciblées + nettoyage + validation.

On génère une checklist priorisée (impact/effort) exportée en Markdown.

In [ ]:
def recommend(err_df: pd.DataFrame) -> list[dict]:
    recs = []

    over_rate = float(err_df.over_translation.mean()) if len(err_df) else 0.0
    under_rate = float(err_df.under_translation.mean()) if len(err_df) else 0.0
    rep_rate = float(err_df.high_repetition.mean()) if len(err_df) else 0.0
    chrf_mean = float(err_df.chrf_sentence.mean()) if len(err_df) else 0.0

    if over_rate > 0.15:
        recs.append({
            "priority": 1,
            "signal": f"over_translation_rate={over_rate:.2%}",
            "action": "Try length control: lower max_new_tokens; increase length_penalty (e.g., 1.1–1.3)",
            "expected_impact": "Medium",
            "effort": "Low",
        })
    if under_rate > 0.15:
        recs.append({
            "priority": 1,
            "signal": f"under_translation_rate={under_rate:.2%}",
            "action": "Allow longer generations: increase max_new_tokens; reduce length_penalty (e.g., 0.8–1.0)",
            "expected_impact": "Medium",
            "effort": "Low",
        })
    if rep_rate > 0.10:
        recs.append({
            "priority": 1,
            "signal": f"high_repetition_rate={rep_rate:.2%}",
            "action": "Add decoding constraints: set no_repeat_ngram_size=3 and re-evaluate", 
            "expected_impact": "Medium",
            "effort": "Low",
        })

    # long sentence weakness heuristic
    long_bucket = err_df[err_df.src_len_bucket.isin(["200-400", ">400"])]
    if len(long_bucket) >= 50:
        long_chrf = float(long_bucket.chrf_sentence.mean())
        if long_chrf + 5 < chrf_mean:
            recs.append({
                "priority": 2,
                "signal": f"long_sentence_chrf={long_chrf:.2f} vs overall={chrf_mean:.2f}",
                "action": "Increase max_source_length; train longer; consider curriculum by length", 
                "expected_impact": "High",
                "effort": "Medium",
            })

    if not recs:
        recs.append({
            "priority": 3,
            "signal": "No strong heuristic flags",
            "action": "Run deeper error audit (linguistic categories, morphology, named entities) + add targeted data",
            "expected_impact": "Medium",
            "effort": "Medium",
        })

    return sorted(recs, key=lambda r: r["priority"])

recs = recommend(err_test)
recs_df = pd.DataFrame(recs)
recs_df

md_lines = ["# Recommandations (auto)", "", f"Run: {run_id}", ""]
for r in recs:
    md_lines.append(f"- P{r['priority']} | impact={r['expected_impact']} | effort={r['effort']} | {r['signal']} → {r['action']}")

(RUN_DIR / "RECOMMENDATIONS.md").write_text("\n".join(md_lines), encoding="utf-8")
print("saved:", RUN_DIR / "RECOMMENDATIONS.md")


## Annexes: analyser les résultats *déjà générés* par les scripts `tools/eval/evaluate_*.py`

Si tu as déjà produit des fichiers dans `docs/results/` (ex: `mt_metrics_ee_to_fr.json`, `mt_preds_ee_to_fr_test.csv`, etc.), tu peux charger et analyser sans réentraîner ici.

In [ ]:
RESULTS_DIR = ROOT / "docs/results"

mt_jsons = list(RESULTS_DIR.glob("mt_metrics_*.json"))
asr_jsons = list(RESULTS_DIR.glob("asr_metrics*.json"))
mt_csvs = list(RESULTS_DIR.glob("mt_preds_*.csv"))
asr_csvs = list(RESULTS_DIR.glob("asr_preds_*.csv"))

print("metrics:", [p.name for p in mt_jsons + asr_jsons])
print("preds:", [p.name for p in mt_csvs[:5]], "... total", len(mt_csvs))

# Load MT metrics
mt_rows = []
for p in mt_jsons:
    try:
        data = json.loads(p.read_text(encoding="utf-8"))
        if isinstance(data, list):
            mt_rows.extend(data)
        elif isinstance(data, dict):
            mt_rows.append(data)
    except Exception as e:
        print("skip", p, e)

mt_metrics_df = pd.DataFrame(mt_rows)
if len(mt_metrics_df):
    display(mt_metrics_df)

# Load ASR metrics
asr_rows = []
for p in asr_jsons:
    try:
        data = json.loads(p.read_text(encoding="utf-8"))
        if isinstance(data, list):
            asr_rows.extend(data)
        elif isinstance(data, dict):
            asr_rows.append(data)
    except Exception as e:
        print("skip", p, e)

asr_metrics_df = pd.DataFrame(asr_rows)
if len(asr_metrics_df):
    display(asr_metrics_df)
